# 05 · Feature research and retraining evidence

This appendix reads the exact completed notebook 02 and 03 records. It distinguishes historical baseline results from new training and keeps unsuccessful experiments visible.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.publication.workflow import evidence, lineage
from march_mania.advanced_features import candidate_blocks
style()
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
FEATURES, FEATURE_RECORD = evidence(ROOT, "feature_store")
MODELS, MODEL_RECORD = evidence(ROOT, "model_comparison")
table(lineage(ROOT))
assert FEATURE_RECORD["summary"]["feature_count"] == MODEL_RECORD["summary"]["feature_count"]
assert FEATURE_RECORD["summary"]["feature_count"] == len(candidate_blocks()["full"])

Stage,Run,Fits,Meaning
02 feature matrix,199b4c00a8b59393a1290727ca6e0e5b9f9bbc5a17db18e1b02354229e5eed89,1050,Feature ablation fits; not the final model search
03 current model search,20e69d30dc746f39f9395f96d1ff13d7a24cb0a7998fd096a0409c31fa7d7193,861,Fits on the exact notebook 02 matrix
Earlier compact research,b0ca1fe8efba1b73a689623d9ae3e6c79de9ff3a1c925a9967e74bce9ae0049d,120,Historical 120-fold study; not current retraining


## Large candidate generation; bounded fitted models

There are 3,106 available candidates, including 2,944 season-local summaries, 26 coach signals and 12 conference-context signals. Per-fold screening, not a global validation-informed filter, caps retained dimensionality at 128. The candidate catalog, full/drop-one ablations and rejection reasons document what was actually tested. Missing women’s coaching sources and rankings remain explicitly unavailable.

In [2]:
screen = pd.read_csv(FEATURES / "screening_summary.csv")
table(screen.query("block == 'full'")[["Gender", "Season", "model", "candidate_count", "retained_count", "rejected_count"]])
feature_scores = pd.read_csv(FEATURES / "leaderboard.csv")
table(feature_scores.sort_values(["Gender", "macro_season_brier"]).groupby(["Gender", "model"], sort=False).head(4)[["Gender", "block", "model", "brier", "macro_season_brier", "games"]])

Gender,Season,model,candidate_count,retained_count,rejected_count
M,2016,hist,3106,128,2978
M,2016,logistic,3106,128,2978
M,2017,hist,3106,128,2978
M,2017,logistic,3106,128,2978
M,2018,hist,3106,128,2978
M,2018,logistic,3106,128,2978
M,2019,hist,3106,128,2978
M,2019,logistic,3106,128,2978
M,2021,hist,3106,128,2978
M,2021,logistic,3106,128,2978


Gender,block,model,brier,macro_season_brier,games
M,rankings,logistic,0.187544,0.187608,334
M,four_factors,logistic,0.187988,0.188058,334
M,dynamic,logistic,0.189077,0.189147,334
M,without_history,hist,0.189634,0.189693,334
M,rankings,hist,0.189703,0.189798,334
M,target_seed,logistic,0.190466,0.190539,334
M,without_form,hist,0.190786,0.190839,334
M,without_conference,hist,0.190903,0.190969,334
W,conference,logistic,0.143331,0.143331,315
W,dynamic,logistic,0.143597,0.143597,315


## Did retraining change predictions?

The comparison pairs old and revised forecasts by population, season, physical game, route and model. Identical seed-reference scores can be correct; a changed feature fingerprint alone is not evidence of better performance. A negative new-minus-old Brier change is better on these matched development games, not proof of leaderboard performance.

In [3]:
comparison = pd.read_csv(MODELS / "comparison.csv")
comparison["brier_change"] = comparison.new_brier - comparison.old_brier
table(comparison[["Gender", "block", "model", "games", "old_brier", "new_brier", "brier_change", "max_probability_change"]])

Gender,block,model,games,old_brier,new_brier,brier_change,max_probability_change
M,M,blend,334,0.192674,0.192674,0.000000,0.000000
M,M,hist_calibrated,334,0.199220,0.199220,0.000000,0.000000
M,M,hist_raw,334,0.199220,0.199220,0.000000,0.000000
M,M,lightgbm_calibrated,334,0.200998,0.200998,0.000000,0.000000
M,M,lightgbm_raw,334,0.200998,0.200998,0.000000,0.000000
M,M,logistic_calibrated,334,0.191740,0.191740,0.000000,0.000000
M,M,logistic_raw,334,0.191358,0.191358,0.000000,0.000000
M,M,rank_logistic_calibrated,334,0.188409,0.188403,-0.000006,0.014947
M,M,rank_logistic_raw,334,0.188409,0.188403,-0.000006,0.014947
M,M,seed_calibrated,334,0.202341,0.202341,0.000000,0.000000


## With and without external ranking information

Men’s ranking logistic, XGBoost and LightGBM candidates are compared with the common/no-Massey model families. Notebook 02 also runs a matched full-versus-without-all-Massey ablation. Women’s and pooled common-feature candidates never consume ranking-derived inputs. Choices and calibration for an outer season use only earlier validation seasons.

In [4]:
scores = pd.read_csv(MODELS / "leaderboard.csv")
ranked = scores.loc[(scores.Gender == "M") & (scores.block == "M") & scores.model.isin(["rank_logistic_raw", "rank_xgboost_raw", "rank_lightgbm_raw", "logistic_raw", "xgboost_raw", "lightgbm_raw"])]
table(ranked[["model", "brier", "macro_season_brier", "log_loss", "roc_auc", "games"]])
intervals = pd.read_csv(FEATURES / "ablation_intervals.csv")
table(intervals.query("Gender == 'M' and baseline == 'without_massey'"))

model,brier,macro_season_brier,log_loss,roc_auc,games
lightgbm_raw,0.200998,0.201110,0.583273,0.755556,334
logistic_raw,0.191358,0.191452,0.558046,0.781421,334
rank_lightgbm_raw,0.192819,0.192901,0.567261,0.779437,334
rank_logistic_raw,0.188403,0.188468,0.554309,0.791703,334
rank_xgboost_raw,0.191462,0.191549,0.564051,0.783189,334
xgboost_raw,0.200264,0.200368,0.582453,0.756656,334


Gender,model,candidate,baseline,season_count,brier_delta,ci_low,ci_high
M,hist,full,without_massey,5,-0.002479,-0.005519,0.000561
M,logistic,full,without_massey,5,0.003250,-0.002029,0.008863


## Supported conclusions and boundaries

Use the measured score changes above, not feature count, to judge the experiment. Selection stability and held-out permutation diagnostics appear in notebooks 02 and 03. Large correlated candidate spaces can overfit a small tournament sample even with nested selection. Five development seasons are limited evidence; 2022–2025 was previously consumed and cannot be relabeled as a new holdout. Notebook 04 evaluates the current logistic anchors and retains the earlier frozen final-prediction release separately. The expanded seeded benchmark regresses from 0.198014/0.138599 to 0.198288/0.146881 for men/women, so the enlarged feature bank is not promoted as a universally better forecaster. Submission generation stays opt-in and never uploads to Kaggle.